# MorphoNet-BT — 06: Main Training Preparation
## Historical objective and execution scope
Prepare the original full-model training pipeline using Architecture v1.0, Preprocessing v1.0 and frozen patient splits. The historical preparation includes metadata checks, one-batch memory tests and one optimizer step on a disposable model to validate AdamW/resume. These operations are NOT safe static checks and must not run in Phase 18. The original preparation excluded full epochs, five-fold training, baselines, ablations and Phase 7; full training was a separate manual cell.

## Public-copy execution policy
This is an English rendering of a historical research notebook. Outputs and execution counts were cleared; no cell was rerun for Phase 18. Historical findings below describe earlier runs only. Every code cell is gated, including setup cells that may perform model checks, preprocessing or writes. Enable `CONFIRM_PUBLIC_WORKFLOW` deliberately in the first code cell only after reviewing [the reproduction guide](../docs/reproducibility.md). Existing stage-specific confirmations must also be enabled separately. Run from the repository root or set `MORPHONET_ROOT` to a private writable experiment workspace with the same source/config layout; set `MORPHONET_DATA_ROOT` for dataset discovery. Original source/configuration fingerprints, private split manifests and checkpoints are not portable artifacts; do not bypass their validation. See the guide for prerequisites and limitations.

**Model identity:** this phase concerns the original full model, later A5. The principal manuscript A4 uses `build_ablation("A4")` and fixed mean fusion, not AMGF.


## Historical validation cells — configuration and environment
The historical SAFE label is not permission to execute these cells during repository preparation.

In [ ]:
CONFIRM_PUBLIC_WORKFLOW = False  # Explicit opt-in; no cell has been rerun.
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
from pathlib import Path
from copy import deepcopy
import sys, json, hashlib, subprocess
PROJECT_ROOT=Path(__import__("os").environ.get("MORPHONET_ROOT", str(Path.cwd()))).resolve()
if PROJECT_ROOT.name=="notebooks": PROJECT_ROOT=PROJECT_ROOT.parent
sys.path.insert(0,str(PROJECT_ROOT))
assert Path(sys.prefix).name.lower()=="dl"
import pandas as pd
import torch
import yaml
from IPython.display import display, Markdown
from src.training.trainer import TRAINING_DEFAULTS,preflight,train_folds
from src.training.setup_validation import batch_size_memory_test
from src.utils.checkpoint import seed_everything
OUTPUT=PROJECT_ROOT/"results/training_setup"
OUTPUT.mkdir(parents=True,exist_ok=True)
config_path=PROJECT_ROOT/"configs/default.yaml"
config_text=config_path.read_text(encoding="utf-8")
config=yaml.safe_load(config_text)
training_already_configured="training" in config
frozen_config={k:v for k,v in config.items() if k!="training"}
config.setdefault("training",deepcopy(TRAINING_DEFAULTS))
seed_everything(config["seed"])
print("Python:",sys.executable)
print("PyTorch:",torch.__version__,"CUDA:",torch.version.cuda)
print("GPU:",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU fallback")
print("Test images will NOT be loaded")

## Architecture fingerprint, splits and patient identifiers
Compare with the Phase 5 fingerprint and check preprocessing/split hashes. Test CSVs are read only to check patient separation; no Test Dataset, DataLoader or performance calculation is involved.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
info=preflight(PROJECT_ROOT,config,1)
display(pd.DataFrame(info["leakage_audit"]))
display(pd.Series(info["coverage"]))
print("Architecture fingerprint:",info["architecture"])
print("Fold 1: train slices",len(info["train_frame"]),"validation slices",len(info["val_frame"]))
(OUTPUT/"preflight_audit.json").write_text(json.dumps({k:v for k,v in info.items() if not isinstance(v,pd.DataFrame)},indent=2),encoding="utf-8")

## Dataset, DataLoader and determinism
Train uses shuffle=True and set_epoch(epoch-1) at each one-based epoch; validation uses shuffle=False and deterministic processing. num_workers=2, persistent_workers=False and drop_last=False. Each loader has a separate generator saved in checkpoints, alongside Python/NumPy/PyTorch/CUDA RNG states. The training loop has no Test loader. Morphology targets and class order remain 0=Meningioma, 1=Glioma, 2=Pituitary. Seed 42; cuDNN deterministic=True, benchmark=False, TF32=False. Bitwise reproducibility across hardware/versions is not guaranteed.
## Losses and weights
CE + 0.3×[Region(BCEWithLogits+Dice) + Boundary(BCEWithLogits+Dice) + 0.5×SmoothL1(Distance)]. Dice is per-sample then averaged, epsilon=1e-6; losses are float32 even with AMP. BCE, Dice and individual tasks, including weighted_morphology_loss, are logged separately. No automatic rebalancing or changing lambdas. class_weights=None. The optional balanced_train mode derives N/(3×count) from Train only after an explicit decision, then averages weighted sample losses; it was not the frozen setting.
## AdamW, scheduler and early stopping
LR=0.001, weight_decay=0.0001; CosineAnnealingLR T_max=100, eta_min=0, one step after each completed epoch. Log used and next LR. Early stopping monitors slice-level validation macro-F1, mode=max, patience=15, min_delta=0; ties neither replace best nor reset patience. Metrics use all epoch predictions, three fixed classes and zero_division=0, not batch-F1 averaging. PID remains available for subsequent aggregation.
## AMP and gradients
CUDA autocast float16 and torch.amp.GradScaler('cuda'); CPU float32 without AMP. Initial scale 1024 is configurable, independent of task weights. Unscale before gradient checks/clipping. No gradient accumulation; max_grad_norm=None. Historically the main trainer stops for nonfinite loss/gradients rather than silently skipping updates; the later baseline overflow repair is documented separately.

## Historical batch-size memory test — not a static test
Sizes 8/16/32 each use one forward/loss/backward pass without optimizer.step, AMP and real fold-1 training morphology targets. num_workers=0 isolates GPU memory from Windows worker startup; the subsequent smoke test uses two workers. Stop increasing size if a conservative projection exceeds 65% of available memory, rather than intentionally reaching OOM. The largest passing size is a recommendation among tested sizes, not the hardware maximum. Adam moments are not allocated before the first step; their additional storage is estimated separately.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
memory,recommended_batch=batch_size_memory_test(PROJECT_ROOT,config,info)
memory.to_csv(OUTPUT/"batch_size_memory_test.csv",index=False)
display(memory)
print("Recommended starting batch size:",recommended_batch)
if not training_already_configured:
    config["training"]["batch_size"]=recommended_batch
else:
    print("Existing explicit training configuration retained; measured recommendation shown above")
assert {k:v for k,v in config.items() if k!="training"}==frozen_config
# Append only a training section on the first setup; never rewrite frozen settings.
if not training_already_configured:
    config_path.write_text(config_text+"\n"+yaml.safe_dump({"training":config["training"]},sort_keys=False),encoding="utf-8")
assert yaml.safe_load(config_path.read_text(encoding="utf-8"))==config
display(config["training"])

## Historical disposable optimizer step and checkpoint/resume check
A separate Windows-compatible process uses one training batch and one AdamW step on a disposable model. It then advances the scheduler, saves a temporary checkpoint and restores into another disposable model. Weights, optimizer, scheduler, AMP, early stopping, RNG and loader generator states are compared before temporary files/models are removed. No best checkpoint, accuracy/F1, training curves or completion marker is saved as an experiment. This is experimental execution and must remain disabled during Phase 18.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
completed=subprocess.run([sys.executable,str(PROJECT_ROOT/"scripts/phase6_smoke.py")],cwd=PROJECT_ROOT,
                         capture_output=True,text=True,encoding="utf-8",errors="replace",timeout=600)
if completed.returncode!=0:
    print(completed.stdout[-3000:]); print(completed.stderr[-6000:])
    raise RuntimeError("Single-batch/resume diagnostic failed: stop and report")
smoke=json.loads((OUTPUT/"training_smoke_test.json").read_text(encoding="utf-8"))
assert smoke["status"]=="PASS"
display(pd.Series(smoke))

## Checkpointing, resume and output protection
Each fold saves best_macro_f1.pt by highest validation macro-F1 and last_checkpoint.pt after every completed epoch. Saving is atomic; last includes a full best snapshot to recover interrupted multi-file writes. Its history is authoritative on resume and reconstructs CSV logs. Checkpoints contain model/optimizer/scheduler/GradScaler, epoch, best metric/epoch, early-stopping state, config/seed/fold, fingerprints, RNG/loader states, metrics and history.
Resume occurs at completed-epoch boundaries. A partial epoch is repeated from the last checkpoint; mid-batch resume is not promised. If interrupted before the first checkpoint, explicitly start anew and archive the partial directory. Configuration, fingerprints and LR/epoch are checked; mismatches stop execution. Hardware/version changes require review. Existing experiments cannot be overwritten by default. resume=True resumes; explicit overwrite=True archives previous results before restarting. Fold locks prevent concurrent runs; after forced termination, verify the process has ended before removing a stale lock.
## Artifacts created only by actual training
`checkpoints/main/fold_X/{best_macro_f1.pt,last_checkpoint.pt}`; `logs/main/fold_X_training.csv` records losses, metrics, LR, time and AMP scale. `results/main/fold_X/` contains metadata, per-epoch Train/Validation predictions with PID, class metrics, best validation predictions and final curves. training_complete.json appears only after successful max_epochs/early stopping completion. Test is not evaluated by these operations.

## Historical final preparation audit and report

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
final_info=preflight(PROJECT_ROOT,config,1)
assert final_info["architecture"]==info["architecture"] and final_info["split_hashes"]==info["split_hashes"]
assert {k:v for k,v in yaml.safe_load(config_path.read_text(encoding="utf-8")).items() if k!="training"}==frozen_config
real_artifacts=list((PROJECT_ROOT/"checkpoints/main").glob("fold_*/*.pt")) if (PROJECT_ROOT/"checkpoints/main").exists() else []
completion_markers=list((PROJECT_ROOT/"results/main").glob("fold_*/training_complete.json")) if (PROJECT_ROOT/"results/main").exists() else []
assert not real_artifacts and not completion_markers, "Unexpected real training artifacts during preparation"
report=f"""Phase 6: MorphoNet-BT training preparation only.
Architecture v1.0, Preprocessing v1.0, splits, seed and labels unchanged. Five-fold PID checks PASS.
Settings: max_epochs=100, batch_size={config['training']['batch_size']}, num_workers=2, persistent_workers=False.
AdamW lr=0.001 weight_decay=0.0001; CosineAnnealingLR T_max=100 eta_min=0.
Early stopping: validation slice-level Macro-F1, max, patience15, min_delta0.
Loss: CE +0.3×(Region BCE+Dice + Boundary BCE+Dice +0.5×SmoothL1).
class_weights=None, max_grad_norm=None, gradient accumulation disabled.
CUDA float16 AMP, GradScaler init_scale1024, float32 losses; CPU fallback without AMP.
One-batch memory checks:
{memory.to_string(index=False)}
Smoke test PASS; finite gradients; one AdamW step on a discarded disposable model, no saved training model.
Model/optimizer/scheduler/scaler/early-stop/RNG/loader restoration passed; changed config and existing outputs rejected.
Each epoch uses set_epoch(epoch-1). Atomic checkpoints resume completed epochs and preserve recoverable best state.
Test stayed closed: CSV metadata for PID checks only, without images, metrics or selection.
No full training, curves, best checkpoint or completion marker; manual training cell excluded from automatic preparation.
One-batch timing does not reliably estimate epoch time; Windows workers/CUDA initialization cannot be multiplied by batch count.
Phase 7 has not started. Manually start fold1 after review, inspect convergence/losses before folds2–5.
"""
(OUTPUT/"training_pipeline_validation.txt").write_text(report,encoding="utf-8")
print(report)

## Deliberate training workflow
Start with fold 1. Review losses, validation macro-F1, region/boundary/distance losses, morphology contribution, epoch time, memory, LR and overfitting before folds 2–5. Do not change lambdas within a run or automatically perform tiny-set overfitting.
## Manual execution required — full training
The next cell starts real training only after explicit confirmation. Set FOLDS_TO_TRAIN=[1] when starting a new study; historical selections below are retained. RESUME=False starts a new run; True restores last_checkpoint. Review settings before enabling CONFIRM_FULL_TRAINING. Do not enable OVERWRITE to bypass an unexplained failure: it archives the previous experiment and starts anew.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
# ============================================================
# MANUAL EXECUTION — FULL TRAINING
# DO NOT AUTO-RUN THIS CELL
# THE RESEARCHER WILL EXECUTE THIS CELL MANUALLY
# ============================================================
FOLDS_TO_TRAIN = [2,3,4,5]
CONFIRM_FULL_TRAINING = False  # Change manually to True only when ready.
RESUME = False
OVERWRITE = False
if CONFIRM_FULL_TRAINING:
    training_results = train_folds(
        PROJECT_ROOT, config, folds=FOLDS_TO_TRAIN,
        confirm_full_training=True, resume=RESUME, overwrite=OVERWRITE,
    )
    display(training_results)
else:
    print("Training has not started. Set CONFIRM_FULL_TRAINING to True manually.")

## After deliberate training
Check training_complete.json and best_macro_f1.pt; review fold 1 before other folds. Smoke-test outcomes are not model performance; never tune training using Test.
**Historical preparation handoff:** five preparation cells were executed; the full-training cell was originally unexecuted with no output. Batch sizes 8/16/32 passed, with 32 recommended. Disposable-model smoke/AMP/resume checks passed; no real training artifacts existed at that preparation stage. Its audit was results/training_setup/manual_cell_audit.json. Later historical notebook execution state may differ; no cells were rerun in Phase 18.